# Day H3 — 판별력, 보정, 임계값: 숫자 셋을 함께 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 참가자 단위 부트스트랩 신뢰구간

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(4)
def make(n):
    X = rng.normal(size=(n, 3))
    y = rng.random(n) < 1 / (1 + np.exp(-(-1.5 + X @ [1.0, 0.6, 0.3])))
    return X, y.astype(int)

X_dev, y_dev = make(800)
X_te, y_te = make(300)                      # 시험 참가자 300명
m = LogisticRegression().fit(X_dev, y_dev)
p = m.predict_proba(X_te)[:, 1]

# 예시 1 — 참가자 단위 부트스트랩 신뢰구간
auc = roc_auc_score(y_te, p)
boot = []
for b in range(2000):
    i = rng.integers(0, len(y_te), len(y_te))       # 참가자를 복원 추출
    if y_te[i].min() == y_te[i].max():              # 한 군만 뽑히면 AUC를 못 구한다
        continue
    boot.append(roc_auc_score(y_te[i], p[i]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"시험 n={len(y_te)}, 사건 {y_te.sum()}명 | AUC {auc:.3f} (95% 부트스트랩 CI {lo:.3f}–{hi:.3f})")

### 예시 2 — AUC는 같고 보정만 다른 세 모델

개발한 모델의 확률을 일부러 변형해 "과신하는" 버전과 "과대 예측하는" 버전을 만듭니다. 순서는 그대로라 AUC는 같습니다.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def calibration(y, p):
    lp = logit(p)
    slope = sm.Logit(y, sm.add_constant(lp)).fit(disp=0).params[1]
    citl = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
    brier = brier_score_loss(y, p)
    scaled = 1 - brier / brier_score_loss(y, np.full(len(y), y.mean()))   # 평균만 맞히는 모델 대비 개선 비율
    return citl, slope, brier, scaled

versions = {"개발한 모델": p,
            "과신(logit×2)": 1 / (1 + np.exp(-2 * logit(p))),
            "과대예측(logit+1)": 1 / (1 + np.exp(-(logit(p) + 1)))}
for name, q in versions.items():
    citl, slope, brier, scaled = calibration(y_te, q)
    print(f"{name:14s} AUC {roc_auc_score(y_te, q):.3f} | 절편 {citl:+.2f} 기울기 {slope:.2f} | "
          f"Brier {brier:.3f} scaled {scaled:+.2f}")

### 예시 3 — 오진 비용 비율로 임계값 정하기

In [ ]:
# 예시 3 — 오진 비용 비율로 임계값 정하기
def expected_cost(y, q, thr, c_fn, c_fp):
    pos = q >= thr
    fn = np.sum(~pos & (y == 1)); fp = np.sum(pos & (y == 0))
    return (c_fn * fn + c_fp * fp) / len(y)

thr_grid = np.round(np.arange(0.05, 0.80, 0.05), 2)
for c_fn, c_fp in [(1, 1), (4, 1), (9, 1)]:          # 놓침(FN) 1건이 위양성(FP) 1건보다 몇 배 나쁜가
    costs = [expected_cost(y_te, p, t, c_fn, c_fp) for t in thr_grid]
    best = thr_grid[int(np.argmin(costs))]
    theory = c_fp / (c_fp + c_fn)
    pos = p >= best
    print(f"FN:FP = {c_fn}:{c_fp} | 이론 임계값 {theory:.2f} | 시험에서 비용 최소 {best:.2f} | "
          f"민감도 {pos[y_te == 1].mean():.2f} 특이도 {(~pos)[y_te == 0].mean():.2f} 양성 비율 {pos.mean():.2f}")

## 직접 해보기

1. `notebooks/advanced/H/H3_performance.ipynb`에서 예시 1–3을 실행합니다.
2. 보정 곡선을 그립니다. 예시 2 다음 셀에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.nonparametric.smoothers_lowess import lowess

out = Path("../../../figures/advanced/H"); out.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(4.8, 4.8))
eng = {"개발한 모델": "developed", "과신(logit×2)": "overconfident", "과대예측(logit+1)": "over-predicting"}
for name, q in versions.items():
    sm_curve = lowess(y_te, q, frac=0.6, return_sorted=True)       # 매끈한 보정 곡선
    ax.plot(sm_curve[:, 0], sm_curve[:, 1], label=eng[name])
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax.set_xlabel("predicted risk"); ax.set_ylabel("observed proportion (lowess)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(out / "H3_calibration.png", dpi=150)
print("saved", out / "H3_calibration.png")